In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(1129)
data = np.load("mnist.npz")
x_train, y_train = data["x_train"], data["y_train"]
x_test, y_test = data["x_test"], data["y_test"]

fig, axes = plt.subplots(2, 5, figsize=(10, 4))
for i, ax in enumerate(axes.ravel()):
    ax.imshow(x_train[i], cmap="gray")
    ax.set_title("label = " + str(y_train[i]))
    ax.axis("off")
plt.tight_layout()
plt.show()
import numpy as np

rng = np.random.default_rng(1129)


class network(object):
    def __init__(self, shape: list, batch_size: int) -> None:
        """
        创建神经网络

        参数：
            shape:神经网络形状
            batch_size:批量
        """

        self.shape = shape
        self.layer_num = len(shape)
        self.batch_size = batch_size

        self.activation = [
            np.zeros((shape[l], batch_size)) for l in range(self.layer_num)
        ]

        self.weights = [
            rng.normal(0, 1, (shape[l + 1], shape[l]))
            for l in range(self.layer_num - 1)
        ]
        self.grad_w = [
            np.zeros((shape[l + 1], shape[l])) for l in range(self.layer_num - 1)
        ]
        self.biases = [
            rng.normal(0, 1, (shape[l + 1], 1)) for l in range(self.layer_num - 1)
        ]

        self.grad_b = [np.zeros((shape[l + 1], 1)) for l in range(self.layer_num - 1)]

        class cache(object):
            def __init__(self, net) -> None:
                self.z = [
                    np.zeros((shape[l], batch_size)) for l in range(net.layer_num - 1)
                ]
                self.d = [
                    np.zeros((shape[l], batch_size)) for l in range(net.layer_num - 1)
                ]
                pass

        self.cache = cache(self)

    def shape_dict(self, l):
        """
        形状字典

        参数：
            l:显示层数
        """
        return {
            f"activation[{l+1}]": np.shape(self.activation[l + 1]),
            f"weights[{l}]": np.shape(self.weights[l]),
            f"activation[{l}]": np.shape(self.activation[l]),
            f"biases[{l}]": np.shape(self.biases[l]),
            f"z[{l}]": np.shape(self.cache.z[l]),
            f"d[{l}]": np.shape(self.cache.d[l]),
            f"d[{l+1}]": np.shape(self.cache.d[l + 1]),
            f"weights[{l+1}]": np.shape(self.weights[l + 1]),
        }

    def rng_input(self, loc=0, scale=1):
        """
        初始化输入(正态分布随机)

        参数：
            loc:均值
            scale:标准差
        """
        self.activation[0] = rng.normal(loc, scale, np.shape(self.activation[0]))

    def rng_ground_truth(self, loc=0.5, scale=0.1):
        self.gt = rng.normal(loc, scale, np.shape(self.activation[-1]))
        pass

    def shape_test(self, l):
        """
        形状字典

        参数：
            l:显示层数
        """
        return {
            f"d[{-1}]": np.shape(self.cache.d[-1]),
            f"grad_w[{-1}]": np.shape(self.grad_w[-1]),
            f"activation[{-1}]": np.shape(self.activation[-1]),
            f"activation[{-2}]": np.shape(self.activation[-2]),
            f"weights[{-l+1}]": np.shape(self.weights[-l + 1]),
            f"d[{-l}]": np.shape(self.cache.d[-l]),
            f"grad_w[{-l}]": np.shape(self.grad_w[-1]),
            f"activation[{-l}]": np.shape(self.activation[-l]),
        }


def sigmoid(x):
    return 1 / (1 + np.exp(-x))


def d_sigmoid(value):
    """
    sigmoid导数

    参数：
        value:sigmoid处理后的值
    """
    return value * (1 - value)


def d_cost(output, ground_truth, batch_size):
    """
    cost的导数

    参数：
        output:输出层数组
        ground_truth:真实值
        batch_size:批量
    """
    return (output - ground_truth) / batch_size


class propagation(object):
    def __init__(self, eta, net: network) -> None:
        self.eta = eta
        self.net = net

    def forward_propagation(self):
        for l in range(self.net.layer_num - 1):
            self.net.cache.z[l] = (
                self.net.weights[l] @ self.net.activation[l] + self.net.biases[l]
            )
            self.net.activation[l + 1] = sigmoid(self.net.cache.z[l])
            self.net.cache.d[l] = d_sigmoid(self.net.activation[l + 1])

    def back_propagation(self):
        d_c = d_cost(self.net.activation[-1], self.net.gt, self.net.batch_size)
        dw = self.net.cache.d[-1] * d_c
        self.net.grad_w[-1] = dw @ self.net.activation[-2].T
        self.net.grad_b[-1] = dw.sum(axis=1, keepdims=True)
        for l in range(2, self.net.layer_num):
            dw = (self.net.weights[-l + 1].T @ dw) * self.net.cache.d[-l]
            self.net.grad_w[-l] = dw @ self.net.activation[-l - 1].T
            self.net.grad_b[-l] = dw.sum(axis=1, keepdims=True)

    def gradient_descent(self):
        for l in range(self.net.layer_num - 1):
            self.net.weights[l] += -self.eta * self.net.grad_w[l]
            self.net.biases[l] += -self.eta * self.net.grad_b[l]
        pass


# ============================================================
# 把 MNIST 变成你的网络能吃的形状
# 你的网络约定：activation[0] 的形状是 (输入维数, 样本数)
#               gt 的形状是 (输出维数, 样本数)，每列是一个 one-hot 标签
# ============================================================


def flatten_normalize(x):
    """(N, 28, 28) 的 uint8 图像 -> (784, N) 的 float,值域缩到 [0, 1]"""
    n = x.shape[0]
    return x.reshape(n, -1).T.astype(np.float64) / 255.0


def one_hot(y, num_classes=10):
    """(N,) 的标签 -> (num_classes, N) 的 one-hot 矩阵，列为样本"""
    n = y.shape[0]
    oh = np.zeros((num_classes, n))
    oh[y, np.arange(n)] = 1.0
    return oh


X_train = flatten_normalize(x_train)
Y_train = one_hot(y_train)
X_test = flatten_normalize(x_test)
Y_test = one_hot(y_test)

print("X_train:", X_train.shape, " Y_train:", Y_train.shape)
print("X_test :", X_test.shape, "  Y_test:", Y_test.shape)


def accuracy(net, X, Y, chunk=200):
    """
    前向传播算预测，与 one-hot 标签比对,返回准确率(0~1)。

    说明：这里只做前向，网络权重/偏置没有被修改，可以放心调用。
    """
    n = X.shape[1]
    correct = 0
    for start in range(0, n, chunk):
        end = min(start + chunk, n)
        a = X[:, start:end]
        for l in range(net.layer_num - 1):
            a = sigmoid(net.weights[l] @ a + net.biases[l])
        correct += np.sum(np.argmax(a, axis=0) == np.argmax(Y[:, start:end], axis=0))
    return correct / n


def train_sgd(net, prop, X, Y, epochs, mini_batch_size, X_eval=None, Y_eval=None):
    """
    在真实数据上训练的核心循环 —— 本次改造的主角。

    原代码的循环条件（最初版 neural_network.ipynb 末尾）：
        times = 100000
        for step in range(times):
            prop.forward_propagation()
            prop.back_propagation()
            prop.gradient_descent()
    问题：
        1. 步数写死成 100000,跑完就结束，不感知"学到什么程度";
        2. 输入和标签都是随机数(rng_input / rng_ground_truth),
           网络没有任何真实的东西可学，跑多少步都没有意义。

    新结构(epoch x mini-batch 双层循环):
        epoch       : 把整个训练集完整过一遍算一轮
        mini-batch  : 每轮内把样本打乱，切成一个个小批，
                      每批做一次 前向 -> 反向 -> 更新
    这是书上从第二章到最后一章所有实验的标准循环骨架，
    之后你要加的每一个改进都长在这个骨架里。

    参数
    ----
    X, Y     : 训练数据，(特征数, 样本数) 与 (类别数, 样本数),Y 是 one-hot
    epochs   : 把整个训练集过几遍
    mini_batch_size : 小批大小（必须与 net.batch_size 一致）
    X_eval, Y_eval  : 可选的评估数据，每轮结束后报告一次准确率
    """
    n = X.shape[1]
    history = {"cost": [], "acc": []}

    for epoch in range(1, epochs + 1):
        # 每轮开头打乱样本顺序（SGD 的标准做法）
        perm = rng.permutation(n)

        cost_sum = 0.0
        num_batches = 0
        for start in range(0, n, mini_batch_size):
            idx = perm[start : start + mini_batch_size]

            # 喂真实数据（原代码这里用的是随机输入 / 随机标签）
            net.activation[0] = X[:, idx]
            net.gt = Y[:, idx]

            prop.forward_propagation()
            prop.back_propagation()
            prop.gradient_descent()

            cost_sum += np.mean((net.activation[-1] - net.gt) ** 2)
            num_batches += 1

        history["cost"].append(cost_sum / num_batches)
        if X_eval is not None:
            acc = accuracy(net, X_eval, Y_eval)
            history["acc"].append(acc)
            print(
                f"epoch {epoch:3d} | cost {history['cost'][-1]:.5f} | test acc {acc:.4f}"
            )
        else:
            print(f"epoch {epoch:3d} | cost {history['cost'][-1]:.5f}")

    return history


# 参考配置：取书中第二章 network.py 的经典取值
#   shape [784, 30, 10]：输入 784，中间 30 个神经元，输出 10
#   eta = 3.0, mini_batch = 10, epochs = 30
# 在这个配置下，训练结束测试准确率大约在 95% 左右。
# 注意：这个 eta 和你最初版里的 [784,128,64,10] / eta=0.05 差很远，
# 换成你的配置会怎样？这正是值得你自己动手实验的地方（见文末）。

shape = [784, 30, 10]
batch_size = 10
eta = 2.5
epochs = 100

net = network(shape, batch_size)
prop = propagation(eta, net)

history = train_sgd(net, prop, X_train, Y_train, epochs, batch_size, X_test, Y_test)
# 画出训练曲线：代价下降 + 测试准确率上升
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history["cost"], marker="o", markersize=3)
axes[0].set_xlabel("epoch")
axes[0].set_ylabel("mean squared cost")
axes[0].set_title("training cost")
axes[0].grid(True, alpha=0.3)

axes[1].plot(history["acc"], marker="o", markersize=3, color="tab:orange")
axes[1].set_xlabel("epoch")
axes[1].set_ylabel("accuracy")
axes[1].set_title("test accuracy")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
